In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
!pip uninstall -y ultralytics
!pip install ultralytics -q

Found existing installation: ultralytics 8.4.36
Uninstalling ultralytics-8.4.36:
  Successfully uninstalled ultralytics-8.4.36


In [4]:
import ultralytics
print("Ultralytics path:", ultralytics.__file__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics path: /usr/local/lib/python3.12/dist-packages/ultralytics/__init__.py


In [5]:
%%writefile custom_modules.py
import torch
import torch.nn as nn

class FPSConv(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv = nn.Conv2d(c, c, 5, padding=2)
        self.bn = nn.BatchNorm2d(c)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class GERB(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 1)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.conv2(self.conv1(x)))

Writing custom_modules.py


In [6]:
from ultralytics import YOLO

model = YOLO("yolov8n.yaml")

In [7]:
from custom_modules import FPSConv

print("Replacing SPPF at 9")

layer = model.model.model[9]
c = layer.cv1.conv.in_channels   # safe

model.model.model[9] = FPSConv(c)

Replacing SPPF at 9


In [8]:
from custom_modules import GERB

# C2f layer indices
c2f_indices = [2, 4, 6, 8, 21]

for idx in c2f_indices:
    print(f"Replacing C2f at layer {idx}")

    layer = model.model.model[idx]

    # Get input channels (IMPORTANT: only c, not c1,c2)
    try:
        c = layer.cv1.conv.in_channels
    except AttributeError:
        c = layer.cv1.conv.in_channels  # fallback

    # Replace with your GERB
    model.model.model[idx] = GERB(c)

Replacing C2f at layer 2
Replacing C2f at layer 4
Replacing C2f at layer 6
Replacing C2f at layer 8
Replacing C2f at layer 21


In [9]:
from custom_modules import GERB

for i in [12, 15, 18]:
    print(f"Replacing C2f at {i}")

    layer = model.model.model[i]
    c = layer.cv1.conv.in_channels

    model.model.model[i] = GERB(c)

Replacing C2f at 12
Replacing C2f at 15
Replacing C2f at 18


In [10]:
for i, layer in enumerate(model.model.model):
    print(i, layer.__class__.__name__)

0 Conv
1 Conv
2 GERB
3 Conv
4 GERB
5 Conv
6 GERB
7 Conv
8 GERB
9 FPSConv
10 Upsample
11 Concat
12 GERB
13 Upsample
14 Concat
15 GERB
16 Conv
17 Concat
18 GERB
19 Conv
20 Concat
21 GERB
22 Detect


In [11]:
%%writefile data.yaml
path: /kaggle/input/datasets/jayeesshv/yolov8mine

train: train/images
val: valid/images
test: test/images

nc: 1
names: ['fire']

Writing data.yaml


In [12]:
model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=32,

    optimizer="SGD",
    lr0=0.01,            # initial learning rate
    momentum=0.937,
    weight_decay=0.0005,

)

Ultralytics 8.4.35 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100, perspective=0.0, plots=True, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7de9f8370a40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [16]:

from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/detect/train2/weights/best.pt")

metrics = model.val(
    data="data.yaml"
)
mp, mr, map50, map = metrics.box.mean_results()

print(f"Precision: {mp:.4f}")
print(f"Recall:    {mr:.4f}")
print(f"mAP50:     {map50:.4f}")
print(f"mAP50-95:  {map:.4f}")


Ultralytics 8.4.35 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 1.4±0.4 ms, read: 78.0±20.5 MB/s, size: 41.2 KB)
val: Scanning /kaggle/input/datasets/jayeesshv/yolov8mine/valid/labels... 96 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 96/96 616.6it/s 0.2s0.2s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/jayeesshv/yolov8mine/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.3it/s 1.8s0.3s
                   all         96         96      0.855      0.801      0.878      0.538
Speed: 3.4ms preprocess, 4.9ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /kaggle/working/runs/detect/val3
Precision: 0.8553
Recall:    0.8008
mAP50:     0.8779
mAP50-95:  0.5375


In [15]:
import shutil

shutil.make_archive(
    '/kaggle/working/runs',  # output zip name
    'zip',
    '/kaggle/working/runs'  # folder to zip
)

'/kaggle/working/runs.zip'

In [12]:
model.train(
    data="data.yaml",
    epochs=200,
    imgsz=640,
    batch=32,

    optimizer="SGD",
    lr0=0.01,            # initial learning rate
    momentum=0.937,
    weight_decay=0.0005,

)

Ultralytics 8.4.36 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100, perspective=0.0, plots=True, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ac7e7286ab0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 